# 04 - Hybrid retrieval

**Goal:** given a question, find the few legal passages most likely to contain the answer.

Two ways to search:

- **Keyword search (BM25)** scores passages by how often the question's *words* appear in them (rare words count more). Great when the question says "Artikel 5" or "Recital 26" - those exact tokens are in the passage heading. Blind to synonyms.
- **Meaning search (embeddings)** compares the question's embedding to every passage's embedding (notebook 03). Great for "when is data anonymous?" even if the law says "no longer identifiable". Weak on exact numbers.

**Hybrid** runs both and merges the two ranked lists. Then we **de-duplicate**: the same provision exists in German and English, and long ones are split into parts - we want each provision only once.

In [1]:
import os
import warnings
from pathlib import Path

from IPython.display import Markdown
from langchain_community.retrievers import BM25Retriever

from assistant.config import get_settings
from assistant.index import chunk_to_documents, load_index, make_embeddings
from assistant.parse import read_chunks
from assistant.retrieve import bm25_tokens, build_retriever, dedupe_cross_language

warnings.filterwarnings("ignore")
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # run from the project root so data/ paths work

settings = get_settings()
chunks = read_chunks(settings.chunks_path)
store = load_index(make_embeddings(settings), settings.chroma_dir)
docs = [d for c in chunks for d in chunk_to_documents(c)]
print(len(chunks), "chunks ->", len(docs), "documents")

/var/folders/0t/jnb24j8d10b9hj3rvp0lf0hm0000gp/T/ipykernel_11397/2780565083.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


W1123 19:10:39.663000 11397 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9952.03it/s]

3044 chunks -> 3638 documents


## How BM25 sees text

BM25 needs the text cut into tokens. Our tokenizer lowercases and splits on anything that is not a letter or digit (umlauts stay). So "Artikel 5" in a question matches "Artikel 5 - Verbotene Praktiken" in a heading.

In [2]:
print(bm25_tokens("Was regelt Artikel 5 der KI-Verordnung?"))
print(bm25_tokens("Art. 6(2) - Größe"))

['was', 'regelt', 'artikel', '5', 'der', 'ki', 'verordnung']
['art', '6', '2', 'größe']


## Three retrievers

- BM25 only, vector (Chroma) only, and the hybrid from `assistant.retrieve` (weights: 0.4 BM25, 0.6 vector).
- To compare fairly, all three get the same de-duplication step and return the top 6 distinct provisions.

In [3]:
K = settings.top_k
bm25 = BM25Retriever.from_documents(docs, k=2 * K, preprocess_func=bm25_tokens)
vector = store.as_retriever(search_kwargs={"k": 2 * K})
hybrid = build_retriever(store, chunks, settings)

def run(name, question, lang):
    if name == "hybrid":
        return hybrid(question, lang)
    raw = (bm25 if name == "BM25" else vector).invoke(question)
    return dedupe_cross_language(raw, K, lang)

def label(d):
    m = d.metadata
    para = f"({m['paragraph']})" if m["paragraph"] else ""
    return f"{m['regulation']} {m['type']} {m['number']}{para} [{m['language']}]"


## Five questions

Each has an **expected** provision. A retriever "finds" it if the provision appears in its top 6; the number is the rank. Matching is on regulation + type + number (+ paragraph/point if given).

In [4]:
QUESTIONS = [
    # (question, language, expected description, matcher)
    ("Was regelt Artikel 5 der KI-Verordnung?", "de", "AI Act Art. 5",
     lambda m: (m["regulation"], m["type"], m["number"]) == ("AI_ACT", "article", "5")),
    ("Wann gelten personenbezogene Daten als anonym?", "de", "GDPR Recital 26",
     lambda m: (m["regulation"], m["type"], m["number"]) == ("GDPR", "recital", "26")),
    ("Is an AI system that screens job applicants' CVs high-risk?", "en",
     "AI Act Art. 6 or Annex III pt 4",
     lambda m: m["regulation"] == "AI_ACT" and (
         (m["type"], m["number"]) == ("article", "6")
         or (m["type"], m["number"], m["paragraph"]) == ("annex", "III", "4"))),
    ("Which AI practices are prohibited under the AI Act?", "en", "AI Act Art. 5",
     lambda m: (m["regulation"], m["type"], m["number"]) == ("AI_ACT", "article", "5")),
    ("When is the processing of personal data lawful?", "en", "GDPR Art. 6",
     lambda m: (m["regulation"], m["type"], m["number"]) == ("GDPR", "article", "6")),
]

def rank_of(results, matcher):
    for i, d in enumerate(results, 1):
        if matcher(d.metadata):
            return i
    return None

rows, details = [], {}
for q, lang, expected, matcher in QUESTIONS:
    row = {"question": q, "expected": expected}
    for name in ("BM25", "vector", "hybrid"):
        results = run(name, q, lang)
        details[(q, name)] = [label(d) for d in results]
        r = rank_of(results, matcher)
        row[name] = f"yes (#{r})" if r else "NO"
    rows.append(row)
table = "| question | expected | BM25 | vector | hybrid |\n|---|---|---|---|---|\n"
table += "\n".join(
    f"| {r['question']} | {r['expected']} | {r['BM25']} | {r['vector']} | {r['hybrid']} |"
    for r in rows
)
Markdown(table)

| question | expected | BM25 | vector | hybrid |
|---|---|---|---|---|
| Was regelt Artikel 5 der KI-Verordnung? | AI Act Art. 5 | yes (#6) | yes (#3) | yes (#4) |
| Wann gelten personenbezogene Daten als anonym? | GDPR Recital 26 | NO | yes (#1) | yes (#1) |
| Is an AI system that screens job applicants' CVs high-risk? | AI Act Art. 6 or Annex III pt 4 | yes (#4) | yes (#1) | yes (#1) |
| Which AI practices are prohibited under the AI Act? | AI Act Art. 5 | yes (#3) | yes (#1) | yes (#1) |
| When is the processing of personal data lawful? | GDPR Art. 6 | yes (#5) | yes (#6) | yes (#3) |

## What is actually in the top 6?

Look at one question in detail. Note the German question only gets German passages - de-duplication keeps the copy in the question's language.

In [5]:
q = QUESTIONS[0][0]
for name in ("BM25", "vector", "hybrid"):
    print(name, "->", details[(q, name)])

BM25 -> ['GDPR recital 93 [de]', 'AI_ACT recital 119 [de]', 'AI_ACT annex VIII(C.5) [de]', 'AI_ACT article 74(5) [de]', 'AI_ACT article 17(1) [de]', 'AI_ACT article 5(1) [de]']
vector -> ['AI_ACT article 3(55) [de]', 'AI_ACT article 1(2) [de]', 'AI_ACT article 5(1) [de]', 'AI_ACT article 57(9) [de]', 'AI_ACT article 58(2) [de]', 'AI_ACT article 57(5) [de]']
hybrid -> ['AI_ACT article 2(12) [de]', 'AI_ACT article 3(55) [de]', 'AI_ACT article 1(2) [de]', 'AI_ACT article 5(1) [de]', 'AI_ACT article 57(9) [de]', 'AI_ACT article 58(2) [de]']


## Reading the table (honestly)

What the run above shows:

- **BM25 alone missed once:** the German anonymisation question never reaches Recital 26, because the recital says "nicht mehr identifiziert werden kann" and never uses the word the question uses. Only meaning search bridges that gap.
- **Even the article-number question is not a clean BM25 win.** Art. 5 shows up at rank 6 for BM25 because "ki", "der" and "verordnung" appear in nearly every passage, so the token "5" is only one weak signal among many. Vector search ranks it #3 and hybrid #4. So in this run the exact-number query is *not* solved by keywords alone; the guaranteed-by-BM25 case is a small corpus (see `tests/test_retrieve.py`).
- **Hybrid is a safety net, not a guaranteed upgrade.** It was never worse than the weaker retriever, equal to the better one on 3 of 5 questions, and best on "When is processing lawful?" (#3 vs #5 and #6) where both lists agreed. On the Art. 5 question it ranked Art. 5 below the vector-only list: three passages that only one list liked outranked it.
- Five questions prove nothing statistically. Stage 6 builds an eval set to measure this properly.
- No stemming: German word forms ("Anbieter" / "Anbietern") do not match in BM25. A possible improvement.